In [1]:
!pip install transformers sentencepiece torch pandas openpyxl tqdm

In [2]:
import pandas as pd
import torch
from transformers import MBartForConditionalGeneration, MBart50TokenizerFast
from tqdm import tqdm

In [3]:
input_file = "eng_updated selected dataset.xlsx"

df = pd.read_excel(input_file)
df.head()

,sentence_id,orginal_sen
0,1,Let's try something.
1,2,I have to go to sleep.
2,3,Today is June 18th and it is Muiriel's birthday!
3,4,Muiriel is 20 now.
4,5,"The password is ""Muiriel""."


In [4]:
english_col = "orginal_sen"

if "sentence_id" not in df.columns:
    df["sentence_id"] = range(1, len(df) + 1)

df = df[["sentence_id", english_col]].copy()
df.rename(columns={english_col: "en_original"}, inplace=True)

df.head()

,sentence_id,en_original
0,1,Let's try something.
1,2,I have to go to sleep.
2,3,Today is June 18th and it is Muiriel's birthday!
3,4,Muiriel is 20 now.
4,5,"The password is ""Muiriel""."


In [6]:
!pip install protobuf sentencepiece

   ---------------------------------------- 0.0/437.9 kB ? eta -:--:--
   ---------------------------------------- 0.0/437.9 kB ? eta -:--:--
    --------------------------------------- 10.2/437.9 kB ? eta -:--:--
    --------------------------------------- 10.2/437.9 kB ? eta -:--:--
   -- ------------------------------------ 30.7/437.9 kB 330.3 kB/s eta 0:00:02
   ------ -------------------------------- 71.7/437.9 kB 438.9 kB/s eta 0:00:01
   ------------------ --------------------- 204.8/437.9 kB 1.0 MB/s eta 0:00:01
   ---------------------------------------- 437.9/437.9 kB 1.8 MB/s eta 0:00:00



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
model_name = "facebook/mbart-large-50-many-to-many-mmt"

tokenizer = MBart50TokenizerFast.from_pretrained(model_name)
model = MBartForConditionalGeneration.from_pretrained(model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

print("mBART model loaded on", device)

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/2.44G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/261 [00:00<?, ?B/s]

mBART model loaded on cpu


In [8]:
def translate_mbart(text, src_lang, tgt_lang):
    tokenizer.src_lang = src_lang
    
    inputs = tokenizer(str(text), return_tensors="pt", truncation=True, padding=True).to(device)

    generated_tokens = model.generate(
        **inputs,
        forced_bos_token_id=tokenizer.lang_code_to_id[tgt_lang]
    )

    return tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)[0]

In [9]:
df_test = df.head(5).copy()

df_test["hi_translation"] = df_test["en_original"].apply(
    lambda x: translate_mbart(x, "en_XX", "hi_IN")
)

df_test["en_backtranslation"] = df_test["hi_translation"].apply(
    lambda x: translate_mbart(x, "hi_IN", "en_XX")
)

df_test

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


,sentence_id,en_original,hi_translation,en_backtranslation
0,1,Let's try something.,चलो कुछ कोशिश करें।,Let 's try something.
1,2,I have to go to sleep.,मुझे सोने जाना है।,I have to go to bed.
2,3,Today is June 18th and it is Muiriel's birthday!,आज 18 जून है और मुइरीएल का जन्मदिन है!,Today is June 18th and Mauriel 's birthday!
3,4,Muiriel is 20 now.,मुइरीएल अब 20 साल का है।,Muriel is now 20 years old.
4,5,"The password is ""Muiriel"".","पासवर्ड ""मुइरिएल"" है।","The password is ""Mauriel""."


In [10]:
tqdm.pandas()

df["hi_translation"] = df["en_original"].progress_apply(
    lambda x: translate_mbart(x, "en_XX", "hi_IN")
)

df.head()

100%|██████████| 200/200 [08:50<00:00,  2.65s/it]


,sentence_id,en_original,hi_translation
0,1,Let's try something.,चलो कुछ कोशिश करें।
1,2,I have to go to sleep.,मुझे सोने जाना है।
2,3,Today is June 18th and it is Muiriel's birthday!,आज 18 जून है और मुइरीएल का जन्मदिन है!
3,4,Muiriel is 20 now.,मुइरीएल अब 20 साल का है।
4,5,"The password is ""Muiriel"".","पासवर्ड ""मुइरिएल"" है।"


In [11]:
df["en_backtranslation"] = df["hi_translation"].progress_apply(
    lambda x: translate_mbart(x, "hi_IN", "en_XX")
)

df.head()

100%|██████████| 200/200 [09:02<00:00,  2.71s/it]


,sentence_id,en_original,hi_translation,en_backtranslation
0,1,Let's try something.,चलो कुछ कोशिश करें।,Let 's try something.
1,2,I have to go to sleep.,मुझे सोने जाना है।,I have to go to bed.
2,3,Today is June 18th and it is Muiriel's birthday!,आज 18 जून है और मुइरीएल का जन्मदिन है!,Today is June 18th and Mauriel 's birthday!
3,4,Muiriel is 20 now.,मुइरीएल अब 20 साल का है।,Muriel is now 20 years old.
4,5,"The password is ""Muiriel"".","पासवर्ड ""मुइरिएल"" है।","The password is ""Mauriel""."


In [12]:
df[["sentence_id", "en_original", "hi_translation", "en_backtranslation"]].head(10)

,sentence_id,en_original,hi_translation,en_backtranslation
0,1,Let's try something.,चलो कुछ कोशिश करें।,Let 's try something.
1,2,I have to go to sleep.,मुझे सोने जाना है।,I have to go to bed.
2,3,Today is June 18th and it is Muiriel's birthday!,आज 18 जून है और मुइरीएल का जन्मदिन है!,Today is June 18th and Mauriel 's birthday!
3,4,Muiriel is 20 now.,मुइरीएल अब 20 साल का है।,Muriel is now 20 years old.
4,5,"The password is ""Muiriel"".","पासवर्ड ""मुइरिएल"" है।","The password is ""Mauriel""."
5,6,I will be back soon.,मैं जल्दी ही लौट जाऊँगा।,I 'll be back soon.
6,7,I'm at a loss for words.,मैं शब्दों के लिए घाटे में हूँ।,I 'm losing words.
7,8,This is never going to end.,यह कभी समाप्त नहीं होगा।,It will never end.
8,9,I just don't know what to say.,मैं बस नहीं जानता कि क्या कहना है।,I just don 't know what to say.
9,10,That was an evil bunny.,यह एक बुरा बछड़ा था।,It was a bad calf.


In [13]:
import re

def clean_text(text):
    if isinstance(text, str):
        return re.sub(r"[\x00-\x1F\x7F]", "", text)
    return text

df = df.applymap(clean_text)

C:\Users\manda\AppData\Local\Temp\ipykernel_1616\1162921249.py:8: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(clean_text)


In [14]:
df.to_excel("mbart_bt.xlsx", index=False)
print("Saved mbart_bt.xlsx")

Saved mbart_bt.xlsx
